In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

26/04/17 18:34:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 18:34:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version: 3.4.2


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import pandas as pd
from datetime import datetime, timedelta

# ============================================================================
# WAREHOUSE PATHS
# ============================================================================
WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")

# SOURCE LAYER PATHS
gold_alerts_path       = f"{WAREHOUSE_ROOT}/gold/alerts"
cases_gold_path        = f"{WAREHOUSE_ROOT}/gold/cases"
tasks_gold_path        = f"{WAREHOUSE_ROOT}/gold/tasks"
transactions_gold_path = f"{WAREHOUSE_ROOT}/gold/transactions"

# METRICS GOLD LAYER PATHS (OUTPUT)
metrics_warehouse      = f"{WAREHOUSE_ROOT}/metrics"
tms_metrics_path       = f"{metrics_warehouse}/transaction_monitoring"
alert_metrics_path     = f"{metrics_warehouse}/alerts"
case_metrics_path      = f"{metrics_warehouse}/cases"
fraud_metrics_path     = f"{metrics_warehouse}/fraud"

print("✓ Warehouse paths configured")
print(f"  Metrics warehouse: {metrics_warehouse}")

✓ Warehouse paths configured
  Metrics warehouse: /opt/Tazama_Warehouse/metrics


In [4]:
# ============================================================================
# LOAD SOURCE TABLES
# ============================================================================
try:
    alerts = spark.read.format("hudi").load(gold_alerts_path)
    cases = spark.read.format("hudi").load(cases_gold_path)
    tasks = spark.read.format("hudi").load(tasks_gold_path)
    transactions = spark.read.format("hudi").load(transactions_gold_path)
    print("✓ All source tables loaded successfully")
except Exception as e:
    print(f"✗ Error loading tables: {e}")

# ============================================================================
# METRIC AGGREGATION UTILITY FUNCTIONS
# ============================================================================

def add_time_dimensions(df, timestamp_col="event_ts"):
    """
    Add standardized time dimensions to support multiple reporting granularities.
    Supports: hourly, daily, monthly, quarterly, annually
    """
    return df.withColumn(
        "metric_date", F.to_date(F.col(timestamp_col))
    ).withColumn(
        "metric_hour", F.date_format(F.col(timestamp_col), "yyyy-MM-dd HH:00:00")
    ).withColumn(
        "metric_day", F.date_format(F.col(timestamp_col), "yyyy-MM-dd")
    ).withColumn(
        "metric_month", F.date_format(F.col(timestamp_col), "yyyy-MM")
    ).withColumn(
        "metric_quarter", F.date_format(
            F.add_months(F.trunc(F.col(timestamp_col), "Q"), 0), "yyyy-Q"
        )
    ).withColumn(
        "metric_year", F.year(F.col(timestamp_col))
    )

def store_metric_gold_layer(df, output_path, metric_name, mode="overwrite"):
    """
    Store pre-aggregated metrics in gold layer with Hudi table format.
    Ensures consistency and efficiency for dashboard consumption.
    """
    try:
        hudi_options = {
            "hoodie.table.name": metric_name,
            "hoodie.datasource.write.operation": "upsert",
            "hoodie.datasource.write.precombine.field": "metric_ts",
            "hoodie.datasource.write.recordkey.field": "metric_id",
            "hoodie.datasource.write.partitionpath.field": "metric_date",
        }
        
        df.write \
            .format("hudi") \
            .mode(mode) \
            .options(**hudi_options) \
            .save(output_path)
        
        print(f"  ✓ {metric_name} stored at {output_path}")
        return True
    except Exception as e:
        print(f"  ✗ Error storing {metric_name}: {e}")
        return False

print("✓ Utility functions defined")

✗ Error loading tables: An error occurred while calling o82.load.
: java.io.FileNotFoundException: File /opt/Tazama_Warehouse/gold/alerts does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hudi.common.util.TablePathUtils.getTablePath(TablePathUtils.java:58)
	at org.apache.hudi.DataSourceUtils.getTablePath(DataSourceUtils.java:79)
	at org.apache.hudi.DefaultSource.createRelation(DefaultSource.scala:111)
	at org.apache.hudi.DefaultSource.createRelation(DefaultSource.scala:74)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:346)
	at org.apache.spark.sql.DataFrameReader.loadV1Source(DataFrameReader.scala:229

26/04/17 18:35:14 WARN DFSPropertiesConfiguration: Cannot find HUDI_CONF_DIR, please set it as the dir of hudi-defaults.conf
26/04/17 18:35:14 WARN DFSPropertiesConfiguration: Properties file file:/etc/hudi/conf/hudi-defaults.conf not found. Ignoring to load props file


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

# Load all tables
alerts = spark.read.format("hudi").load(gold_alerts_path)
cases = spark.read.format("hudi").load(cases_gold_path)
tasks = spark.read.format("hudi").load(tasks_gold_path)
transactions = spark.read.format("hudi").load(transactions_gold_path)

print("="*80)
print("CASE MANAGEMENT METRICS")
print("="*80)

# ============================================================================
# 1. Cases created from alerts
# ============================================================================
cases_from_alerts = cases.filter(F.col("case_creation_type") == "AUTOMATIC_SYSTEM").count()

# ============================================================================
# 2. Cases manually created (no alert)
# ============================================================================
manual_cases = cases.filter(F.col("case_creation_type") != "AUTOMATIC_SYSTEM").count()
print(manual_cases)
# ============================================================================
# 3. Alerts (cases) automatically closed by ATM
# ============================================================================
# Auto-close conditions:
# 1. prediction_outcome_norm = FALSE_POSITIVE, OR
# 2. alert_type_norm = FRAUD AND prediction_outcome_norm = TRUE_POSITIVE AND transaction_status = BLOCKED

# Join alerts with transactions to get transaction status
alerts_with_tx_status = alerts.join(
    transactions.select(
        F.col("tx_msg_id"),
        F.col("tx_status").alias("transaction_status")
    ),
    "tx_msg_id",
    "left"
)

auto_closed_cases = alerts_with_tx_status.filter(
    (F.col("prediction_outcome_norm") == "FALSE_POSITIVE") |
    (
        (F.col("alert_type_norm") == "FRAUD") &
        (F.col("prediction_outcome_norm") == "TRUE_POSITIVE") &
        (F.col("transaction_status") == "BLOCKED")
    )
).select("case_id").distinct().count()

# ============================================================================
# 4. Auto-Closure Rate
# ============================================================================
total_alerts_generated = alerts.select("alert_id").distinct().count()
auto_closure_rate = (auto_closed_cases / total_alerts_generated * 100) if total_alerts_generated > 0 else 0

# ============================================================================
# 5. Auto-close cases reopened
# ============================================================================
auto_closed_case_ids = alerts_with_tx_status.filter(
    (F.col("prediction_outcome_norm") == "FALSE_POSITIVE") |
    (
        (F.col("alert_type_norm") == "FRAUD") &
        (F.col("prediction_outcome_norm") == "TRUE_POSITIVE") &
        (F.col("transaction_status") == "BLOCKED")
    )
).select("case_id").distinct()

auto_close_reopened = cases.join(auto_closed_case_ids, "case_id", "inner") \
    .filter(F.col("has_parent_case") == 1).count()

# ============================================================================
# 6. Auto-Closure Reopen Rate
# ============================================================================
auto_closure_reopen_rate = (auto_close_reopened / auto_closed_cases * 100) if auto_closed_cases > 0 else 0

# ============================================================================
# 7. Manual Case Rate
# ============================================================================
total_cases = cases.count()
manual_case_rate = (manual_cases / total_cases * 100) if total_cases > 0 else 0

# ============================================================================
# 8. Reopened Case Rate
# ============================================================================
cases_reopened = cases.filter(F.col("has_parent_case") == 1).count()
cases_closed = cases.filter(
    F.col("status").like("%CLOSED%") | 
    F.col("status").like("%COMPLETE%") |
    F.col("status").like("%DISPOSED%")
).count()
reopened_case_rate = (cases_reopened / cases_closed * 100) if cases_closed > 0 else 0

# ============================================================================
# 9, 10, 11. AML, Fraud, and Fraud+AML cases
# ============================================================================
cases_with_type = cases.join(
    alerts.select("case_id", "alert_type_norm").distinct(),
    "case_id",
    "left"
)

aml_cases = cases_with_type.filter(F.col("alert_type_norm") == "AML") \
    .select("case_id").distinct().count()
fraud_cases = cases_with_type.filter(F.col("alert_type_norm") == "FRAUD") \
    .select("case_id").distinct().count()
fraud_aml_cases = cases_with_type.filter(F.col("alert_type_norm") == "FRAUD_AND_AML") \
    .select("case_id").distinct().count()

# ============================================================================
# 12. Open investigations
# ============================================================================
open_investigations = cases.join(
    tasks.filter(
        F.col("status") == "STATUS_20_IN_PROGRESS"
    ).select("case_id").distinct(),
    "case_id",
    "inner"
).select("case_id").distinct().count()

# ============================================================================
# 13. Investigation tasks by status
# ============================================================================
investigation_tasks_by_status = tasks.filter(
    F.col("status") == "STATUS_20_IN_PROGRESS"
).groupBy("status").agg(
    F.count("task_id").alias("task_count")
).orderBy(F.desc("task_count"))

# ============================================================================
# 14. Investigation Conversion Rate
# ============================================================================
cases_with_investigate_task = cases.join(
    tasks.filter(
        F.col("status") == "STATUS_20_IN_PROGRESS"
    ).select("case_id").distinct(),
    "case_id",
    "inner"
).select("case_id").distinct().count()

investigation_conversion_rate = (cases_with_investigate_task / cases_from_alerts * 100) \
    if cases_from_alerts > 0 else 0

# ============================================================================
# 15. Cases closed
# ============================================================================
# Already calculated as cases_closed above

# ============================================================================
# 16. Cases by status & disposition
# ============================================================================
cases_by_status = cases.groupBy("status").agg(
    F.count("case_id").alias("case_count")
).orderBy(F.desc("case_count"))

# Cases by disposition (using actual case status codes)
confirmed_cases = cases.filter(
    F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
).select("case_id").distinct().count()

refuted_cases = cases.filter(
    F.col("status") == "STATUS_81_CLOSED_REFUTED"
).select("case_id").distinct().count()

inconclusive_cases = cases.filter(
    F.col("status") == "STATUS_83_CLOSED_INCONCLUSIVE"
).select("case_id").distinct().count()

# ============================================================================
# 17. Tasks metrics
# ============================================================================
open_tasks = tasks.filter(F.col("is_completed") == 0).count()

tasks_by_status = tasks.groupBy("status").agg(
    F.count("task_id").alias("task_count")
).orderBy(F.desc("task_count"))

open_tasks_by_type = tasks.filter(F.col("is_completed") == 0).groupBy("candidate_group").agg(
    F.count("task_id").alias("task_count")
).orderBy(F.desc("task_count"))

# ============================================================================
# 18. SAR/STR filed - NOT AVAILABLE in current schema
# ============================================================================
sar_str_filed = "N/A - Field not available"

# ============================================================================
# 19 & 20. Case to Disposition Time / MTTD
# ============================================================================
cases_with_duration = cases.filter(F.col("created_to_updated_ms") > 0)
avg_case_duration_ms = cases_with_duration.agg(F.avg("created_to_updated_ms")).collect()[0][0]
mttd_hours = (avg_case_duration_ms / (1000 * 60 * 60)) if avg_case_duration_ms else 0
mttd_days = mttd_hours / 24

# ============================================================================
# 21. MTTD by disposition
# ============================================================================
mttd_by_disposition = cases.join(
    alerts.select("case_id", "prediction_outcome_norm").distinct(),
    "case_id",
    "left"
).filter(
    F.col("created_to_updated_ms") > 0
).groupBy("prediction_outcome_norm").agg(
    F.avg("created_to_updated_ms").alias("avg_duration_ms"),
    F.count("case_id").alias("case_count"))
mttd_by_disposition = cases.filter(
    (F.col("created_to_updated_ms") > 0) &
    F.col("status").isin(
        "STATUS_81_CLOSED_REFUTED",
        "STATUS_82_CLOSED_CONFIRMED",
        "STATUS_83_CLOSED_INCONCLUSIVE"
    )
).groupBy("status").agg(
    F.avg("created_to_updated_ms").alias("avg_duration_ms"),
    F.count("case_id").alias("case_count")
).withColumn(
    "avg_duration_hours",
    F.round(F.col("avg_duration_ms") / (1000 * 60 * 60), 2)
).select("status", "case_count", "avg_duration_hours") \
 .orderBy("status")

# ============================================================================
# 22. Confirmation Rate
# ============================================================================
confirmation_rate = (confirmed_cases / cases_closed * 100) if cases_closed > 0 else 0

# ============================================================================
# 23. Case Closure Rate per investigator
# ============================================================================
cases_with_owner = cases.filter(F.col("has_owner") == 1).count()

cases_per_owner = cases.filter(F.col("case_owner_user_id").isNotNull()) \
    .groupBy("case_owner_user_id").agg(
        F.count("case_id").alias("total_cases"),
        F.sum(F.when(
            F.col("status").like("%CLOSED%") | F.col("status").like("%COMPLETE%"), 
            1
        ).otherwise(0)).alias("closed_cases")
    ).withColumn(
        "closure_rate_pct",
        F.round((F.col("closed_cases") / F.col("total_cases")) * 100, 2)
    ).orderBy(F.desc("total_cases"))

print("\n✅ Case metrics calculation complete!")
print(f"   Total cases analyzed: {total_cases}")
print(f"   Total alerts analyzed: {total_alerts_generated}")

Py4JJavaError: An error occurred while calling o85.load.
: java.io.FileNotFoundException: File /opt/Tazama_Warehouse/gold/alerts does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hudi.common.util.TablePathUtils.getTablePath(TablePathUtils.java:58)
	at org.apache.hudi.DataSourceUtils.getTablePath(DataSourceUtils.java:79)
	at org.apache.hudi.DefaultSource.createRelation(DefaultSource.scala:111)
	at org.apache.hudi.DefaultSource.createRelation(DefaultSource.scala:74)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:346)
	at org.apache.spark.sql.DataFrameReader.loadV1Source(DataFrameReader.scala:229)
	at org

In [18]:
# Create Case Metrics Summary DataFrame
case_metrics_df = pd.DataFrame({
    "Metric": [
        "1. Cases from Alerts",
        "2. Manual Cases",
        "3. Auto-Closed Cases",
        "4. Auto-Closure Rate (%)",
        "5. Auto-Close Reopened",
        "6. Auto-Closure Reopen Rate (%)",
        "7. Manual Case Rate (%)",
        "8. Reopened Case Rate (%)",
        "9. AML Cases",
        "10. Fraud Cases",
        "11. Fraud & AML Cases",
        "12. Open Investigations",
        "14. Investigation Conversion Rate (%)",
        "15. Cases Closed",
        "16a. Confirmed Cases (Closed)",
        "16b. Refuted Cases (Closed)",
        "16c. Inconclusive Cases (Closed)",
        "17. Open Tasks",
        "19-20. MTTD (hours)",
        "22. Confirmation Rate (%)",
        "23. Cases with Owner"
    ],
    "Value": [
        cases_from_alerts,
        manual_cases,
        auto_closed_cases,
        round(auto_closure_rate, 2),
        auto_close_reopened,
        round(auto_closure_reopen_rate, 2),
        round(manual_case_rate, 2),
        round(reopened_case_rate, 2),
        aml_cases,
        fraud_cases,
        fraud_aml_cases,
        open_investigations,
        round(investigation_conversion_rate, 2),
        cases_closed,
        confirmed_cases,
        refuted_cases,
        inconclusive_cases,
        open_tasks,
        round(mttd_hours, 2),
        round(confirmation_rate, 2),
        cases_with_owner
    ]
})

print("\n" + "="*80)
print("CASE METRICS SUMMARY")
print("="*80)
print(case_metrics_df.to_string(index=False))

print("\n" + "="*80)
print("DETAILED BREAKDOWNS")
print("="*80)

print("\n13. Investigation Tasks by Status:")
investigation_tasks_by_status.show(truncate=False)

print("\n16. Cases by Status:")
cases_by_status.show(truncate=False)

print("\n17a. Tasks by Status:")
tasks_by_status.show(truncate=False)

print("\n17b. Open Tasks by Type:")
open_tasks_by_type.show(truncate=False)

print("\n21. MTTD by Disposition:")
mttd_by_disposition.show(truncate=False)

print("\n23. Case Closure Rate per Investigator:")
cases_per_owner.show(10, truncate=False)

print("\n" + "="*80)
print("MISSING METRICS")
print("="*80)
print(f"18. SAR/STR Filed: {sar_str_filed}")
print("="*80)


CASE METRICS SUMMARY
                               Metric    Value
                 1. Cases from Alerts 128381.0
                      2. Manual Cases      0.0
                 3. Auto-Closed Cases  32095.0
             4. Auto-Closure Rate (%)     25.0
               5. Auto-Close Reopened      0.0
      6. Auto-Closure Reopen Rate (%)      0.0
              7. Manual Case Rate (%)      0.0
            8. Reopened Case Rate (%)      0.0
                         9. AML Cases      0.0
                      10. Fraud Cases      0.0
                11. Fraud & AML Cases      0.0
              12. Open Investigations      0.0
14. Investigation Conversion Rate (%)      0.0
                     15. Cases Closed      0.0
        16a. Confirmed Cases (Closed)      0.0
          16b. Refuted Cases (Closed)      0.0
     16c. Inconclusive Cases (Closed)      0.0
                       17. Open Tasks 128378.0
                  19-20. MTTD (hours)      0.0
            22. Confirmation Rate (%) 

In [19]:
# ============================================================================
# SECTION 2.3: CASE METRICS - GOLD LAYER
# Pre-Aggregated at Multiple Granularities (Hourly, Daily, Monthly, Quarterly, Annually)
# ============================================================================

print("\n" + "="*80)
print("2.3 CASE METRICS - GOLD LAYER")
print("="*80)

# Prepare cases with time dimensions
cases_with_time = add_time_dimensions(cases, timestamp_col="case_created_ts")
tasks_with_time = add_time_dimensions(tasks, timestamp_col="task_created_ts")

# METRIC 1: Cases created from alerts (automatic)
cases_from_alerts_daily = cases_with_time.filter(
    F.col("case_creation_type") == "AUTOMATIC_SYSTEM"
).groupBy("metric_date").agg(
    F.count("case_id").alias("cases_from_alerts")
)

# METRIC 2: Manually created cases
manual_cases_daily = cases_with_time.filter(
    F.col("case_creation_type") != "AUTOMATIC_SYSTEM"
).groupBy("metric_date").agg(
    F.count("case_id").alias("manual_cases_created")
)

# METRIC 3 & 4: Auto-closed cases and auto-closure rate
auto_closed_cases = cases_with_time.filter(
    (F.col("status").like("%CLOSED%") | F.col("status").like("%AUTO%"))
).groupBy("metric_date").agg(
    F.count("case_id").alias("auto_closed_cases")
)

# METRIC 5: Auto-closure reopened
auto_closed_reopened = cases_with_time.filter(
    (F.col("status").like("%CLOSED%")) & (F.col("has_parent_case") == 1)
).groupBy("metric_date").agg(
    F.count("case_id").alias("auto_closed_reopened")
)

# METRIC 6: Reopened case rate
cases_reopened = cases_with_time.filter(
    F.col("has_parent_case") == 1
).groupBy("metric_date").agg(
    F.count("case_id").alias("cases_reopened")
)

cases_closed_total = cases_with_time.filter(
    F.col("status").like("%CLOSED%") | F.col("status").like("%COMPLETE%")
).groupBy("metric_date").agg(
    F.count("case_id").alias("total_cases_closed")
)

# METRIC 7: Manual case rate
total_cases_daily = cases_with_time.groupBy("metric_date").agg(
    F.count("case_id").alias("total_cases_created")
)

# METRIC 9, 10, 11: AML, Fraud, and combined cases
cases_with_alert_type = cases_with_time.join(
    alerts.select("case_id", "alert_type_norm").distinct(),
    "case_id",
    "left"
)

aml_cases_daily = cases_with_alert_type.filter(
    F.col("alert_type_norm") == "AML"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("aml_cases")
)

fraud_cases_daily = cases_with_alert_type.filter(
    F.col("alert_type_norm") == "FRAUD"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("fraud_cases")
)

fraud_aml_cases_daily = cases_with_alert_type.filter(
    F.col("alert_type_norm") == "FRAUD_AND_AML"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("fraud_aml_cases")
)

# METRIC 12: Open investigations
open_investigations_daily = cases_with_time.join(
    tasks_with_time.filter(F.col("status") == "STATUS_20_IN_PROGRESS").select("case_id").distinct(),
    "case_id",
    "inner"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("cases_with_open_investigations")
)

# METRIC 13: Investigation tasks by status
investigation_tasks_by_status = tasks_with_time.groupBy("metric_date", "status").agg(
    F.count("task_id").alias("task_count")
)

# METRIC 14: Investigation conversion rate (placeholder)
cases_with_investigate = cases_with_time.join(
    tasks_with_time.filter(F.col("status") == "STATUS_20_IN_PROGRESS").select("case_id").distinct(),
    "case_id",
    "inner"
).groupBy("metric_date").agg(
    F.count("case_id").alias("cases_with_investigate_tasks")
)

# METRIC 15: Cases closed (already calculated)
# METRIC 16: Cases by status
cases_by_status = cases_with_time.groupBy("metric_date", "status").agg(
    F.count("case_id").alias("case_count")
)

# METRIC 16a, 16b, 16c: Cases by disposition
confirmed_cases = cases_with_time.filter(
    F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
).groupBy("metric_date").agg(
    F.count("case_id").alias("confirmed_cases")
)

refuted_cases = cases_with_time.filter(
    F.col("status") == "STATUS_81_CLOSED_REFUTED"
).groupBy("metric_date").agg(
    F.count("case_id").alias("refuted_cases")
)

inconclusive_cases = cases_with_time.filter(
    F.col("status") == "STATUS_83_CLOSED_INCONCLUSIVE"
).groupBy("metric_date").agg(
    F.count("case_id").alias("inconclusive_cases")
)

# METRIC 17: Open tasks
open_tasks_daily = tasks_with_time.filter(
    F.col("is_completed") == 0
).groupBy("metric_date").agg(
    F.count("task_id").alias("open_tasks")
)

# METRIC 17b: Open tasks by type
open_tasks_by_type = tasks_with_time.filter(
    F.col("is_completed") == 0
).groupBy("metric_date", "candidate_group").agg(
    F.count("task_id").alias("open_task_count")
)

# METRIC 19 & 20: Case Duration (MTTD)
cases_with_duration = cases_with_time.filter(
    F.col("created_to_updated_ms") > 0
).groupBy("metric_date").agg(
    F.avg("created_to_updated_ms").alias("avg_case_duration_ms"),
    F.count("case_id").alias("cases_with_duration")
).withColumn(
    "avg_mttd_hours",
    F.round(F.col("avg_case_duration_ms") / (1000 * 60 * 60), 2)
).withColumn(
    "avg_mttd_days",
    F.round(F.col("avg_case_duration_ms") / (1000 * 60 * 60 * 24), 2)
)

# METRIC 22: Confirmation rate
# METRIC 23: Case closure rate per investigator
cases_per_investigator = cases_with_time.filter(
    F.col("case_owner_user_id").isNotNull()
).groupBy("metric_date", "case_owner_user_id").agg(
    F.count("case_id").alias("total_cases_owner"),
    F.sum(F.when(F.col("status").like("%CLOSED%"), 1).otherwise(0)).alias("closed_cases_owner")
).withColumn(
    "investigator_closure_rate_pct",
    F.round((F.col("closed_cases_owner") / F.col("total_cases_owner")) * 100, 2)
)

# Combine all case metrics
case_metrics_daily = total_cases_daily.join(
    cases_from_alerts_daily, "metric_date", "left"
).join(
    manual_cases_daily, "metric_date", "left"
).join(
    auto_closed_cases, "metric_date", "left"
).join(
    auto_closed_reopened, "metric_date", "left"
).join(
    cases_reopened, "metric_date", "left"
).join(
    cases_closed_total, "metric_date", "left"
).join(
    aml_cases_daily, "metric_date", "left"
).join(
    fraud_cases_daily, "metric_date", "left"
).join(
    fraud_aml_cases_daily, "metric_date", "left"
).join(
    open_investigations_daily, "metric_date", "left"
).join(
    cases_with_investigate, "metric_date", "left"
).join(
    confirmed_cases, "metric_date", "left"
).join(
    refuted_cases, "metric_date", "left"
).join(
    inconclusive_cases, "metric_date", "left"
).join(
    open_tasks_daily, "metric_date", "left"
).join(
    cases_with_duration, "metric_date", "left"
).fillna(0)

# Add calculated rates
case_metrics_daily = case_metrics_daily.withColumn(
    "auto_closure_rate_pct",
    F.when(
        F.col("cases_from_alerts") > 0,
        F.round((F.col("auto_closed_cases") / F.col("cases_from_alerts")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "auto_closure_reopen_rate_pct",
    F.when(
        F.col("auto_closed_cases") > 0,
        F.round((F.col("auto_closed_reopened") / F.col("auto_closed_cases")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "manual_case_rate_pct",
    F.when(
        F.col("total_cases_created") > 0,
        F.round((F.col("manual_cases_created") / F.col("total_cases_created")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "reopened_case_rate_pct",
    F.when(
        F.col("total_cases_closed") > 0,
        F.round((F.col("cases_reopened") / F.col("total_cases_closed")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "investigation_conversion_rate_pct",
    F.when(
        F.col("cases_from_alerts") > 0,
        F.round((F.col("cases_with_investigate_tasks") / F.col("cases_from_alerts")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "confirmation_rate_pct",
    F.when(
        F.col("total_cases_closed") > 0,
        F.round((F.col("confirmed_cases") / F.col("total_cases_closed")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "metric_ts", F.lit(datetime.now())
)

print("\n✓ Case Metrics Calculated (23 metrics):")
for i, metric in enumerate([
    "Cases from Alerts",
    "Manually Created Cases",
    "Auto-Closed Cases",
    "Auto-Closure Rate",
    "Auto-Closure Reopened",
    "Auto-Closure Reopen Rate",
    "Manual Case Rate",
    "Reopened Case Rate",
    "AML Cases",
    "Fraud Cases",
    "Fraud & AML Cases",
    "Open Investigations",
    "Investigation Tasks by Status",
    "Investigation Conversion Rate",
    "Cases Closed",
    "Cases by Status",
    "Confirmed Cases",
    "Refuted Cases",
    "Inconclusive Cases",
    "Open Tasks",
    "Tasks by Status",
    "Case Duration (MTTD)",
    "Case Closure Rate per Investigator"
], 1):
    print(f"  Metric {i}: {metric}")

# Display sample data
print("\nSample Case Metrics (Daily Granularity):")
case_metrics_daily.select(
    "metric_date", "total_cases_created", "cases_from_alerts", "manual_cases_created",
    "auto_closed_cases", "auto_closure_rate_pct", "confirmed_cases", "avg_mttd_days"
).limit(10).show(truncate=False)


2.3 CASE METRICS - GOLD LAYER

✓ Case Metrics Calculated (23 metrics):
  Metric 1: Cases from Alerts
  Metric 2: Manually Created Cases
  Metric 3: Auto-Closed Cases
  Metric 4: Auto-Closure Rate
  Metric 5: Auto-Closure Reopened
  Metric 6: Auto-Closure Reopen Rate
  Metric 7: Manual Case Rate
  Metric 8: Reopened Case Rate
  Metric 9: AML Cases
  Metric 10: Fraud Cases
  Metric 11: Fraud & AML Cases
  Metric 12: Open Investigations
  Metric 13: Investigation Tasks by Status
  Metric 14: Investigation Conversion Rate
  Metric 15: Cases Closed
  Metric 16: Cases by Status
  Metric 17: Confirmed Cases
  Metric 18: Refuted Cases
  Metric 19: Inconclusive Cases
  Metric 20: Open Tasks
  Metric 21: Tasks by Status
  Metric 22: Case Duration (MTTD)
  Metric 23: Case Closure Rate per Investigator

Sample Case Metrics (Daily Granularity):
+-----------+-------------------+-----------------+--------------------+-----------------+---------------------+---------------+-------------+
|metric_date

In [20]:
# ============================================================================
# SECTION 2.4: FRAUD METRICS - GOLD LAYER
# Pre-Aggregated at Multiple Granularities (Hourly, Daily, Monthly, Quarterly, Annually)
# ============================================================================

print("\n" + "="*80)
print("2.4 FRAUD METRICS - GOLD LAYER")
print("="*80)

# PREREQUISITE: Define variables from TMS and Alert sections if not already defined
# ============================================================================
if 'tx_with_time' not in locals():
    tx_with_time = add_time_dimensions(transactions, timestamp_col="ingested_at_ts")
    
if 'tx_evaluated' not in locals():
    tx_evaluated = tx_with_time.join(
        alerts.select("tx_msg_id").distinct(),
        tx_with_time.tx_msg_id == alerts.tx_msg_id,
        "inner"
    ).select(tx_with_time["*"])
    
if 'alerts_with_time' not in locals():
    alerts_with_time = add_time_dimensions(alerts, timestamp_col="created_at_ts")
    
if 'cases_with_time' not in locals():
    cases_with_time = add_time_dimensions(cases, timestamp_col="case_created_ts")

# Join all relevant tables
fraud_base = cases_with_time.join(
    alerts.select("case_id", "alert_type_norm", "prediction_outcome_norm", "tx_msg_id").distinct(),
    "case_id",
    "left"
).join(
    transactions.select("tx_msg_id", "tx_amount", "tx_status", "event_ts"),
    "tx_msg_id",
    "left"
)

# METRIC 1-3: Case outcomes by disposition
confirmed_fraud_daily = fraud_base.filter(
    F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
).groupBy("metric_date").agg(
    F.count("case_id").alias("confirmed_cases"),
    F.sum(F.when(F.col("tx_amount").isNotNull(), F.col("tx_amount")).otherwise(0)).alias("confirmed_fraud_value")
)

refuted_fraud_daily = fraud_base.filter(
    F.col("status") == "STATUS_81_CLOSED_REFUTED"
).groupBy("metric_date").agg(
    F.count("case_id").alias("refuted_cases")
)

inconclusive_fraud_daily = fraud_base.filter(
    F.col("status") == "STATUS_83_CLOSED_INCONCLUSIVE"
).groupBy("metric_date").agg(
    F.count("case_id").alias("inconclusive_cases")
)

# METRIC 4-6: Detection accuracy using predictions
tp_from_alerts = fraud_base.filter(
    F.col("prediction_outcome_norm") == "TRUE_POSITIVE"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("true_positives")
)

fp_from_alerts = fraud_base.filter(
    F.col("prediction_outcome_norm") == "FALSE_POSITIVE"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("false_positives")
)

fn_from_alerts = fraud_base.filter(
    F.col("prediction_outcome_norm") == "FALSE_NEGATIVE"
).groupBy("metric_date").agg(
    F.countDistinct("case_id").alias("false_negatives")
)

# Combine fraud metrics base table
fraud_metrics_daily = confirmed_fraud_daily.join(
    refuted_fraud_daily, "metric_date", "left"
).join(
    inconclusive_fraud_daily, "metric_date", "left"
).join(
    tp_from_alerts, "metric_date", "left"
).join(
    fp_from_alerts, "metric_date", "left"
).join(
    fn_from_alerts, "metric_date", "left"
).fillna(0)

# METRIC 7-9: Model Performance Metrics
fraud_metrics_daily = fraud_metrics_daily.withColumn(
    "precision",
    F.when(
        (F.col("true_positives") + F.col("false_positives")) > 0,
        F.round(F.col("true_positives") / (F.col("true_positives") + F.col("false_positives")), 4)
    ).otherwise(0)
).withColumn(
    "recall_tpr",
    F.when(
        (F.col("true_positives") + F.col("false_negatives")) > 0,
        F.round(F.col("true_positives") / (F.col("true_positives") + F.col("false_negatives")), 4)
    ).otherwise(0)
).withColumn(
    "f1_score",
    F.when(
        (F.col("precision") + F.col("recall_tpr")) > 0,
        F.round(2 * (F.col("precision") * F.col("recall_tpr")) / (F.col("precision") + F.col("recall_tpr")), 4)
    ).otherwise(0)
)

# Get total transactions evaluated for rate calculations
evaluated_tx_fraud = tx_evaluated.groupBy(
    F.date_format(F.col("ingested_at_ts"), "yyyy-MM-dd").alias("metric_date")
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_evaluated")
)

transactions_with_alerts_fraud = alerts_with_time.groupBy("metric_date").agg(
    F.countDistinct("tx_msg_id").alias("transactions_with_alerts")
)

# METRIC 10-11: Error rates
fraud_metrics_daily = fraud_metrics_daily.join(
    transactions_with_alerts_fraud, "metric_date", "left"
).join(
    evaluated_tx_fraud, "metric_date", "left"
).fillna(0)

fraud_metrics_daily = fraud_metrics_daily.withColumn(
    "false_positive_rate_pct",
    F.when(
        F.col("transactions_with_alerts") > 0,
        F.round((F.col("false_positives") / F.col("transactions_with_alerts")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "false_negative_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.col("false_negatives") / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
)

# METRIC 12: Prevented fraud value (interdicted confirmed fraud)
prevented_fraud_daily = fraud_base.filter(
    (F.col("status") == "STATUS_82_CLOSED_CONFIRMED") &
    (F.col("tx_status").isin("BLOCKED", "REJECTED"))
).groupBy("metric_date").agg(
    F.sum(F.when(F.col("tx_amount").isNotNull(), F.col("tx_amount")).otherwise(0)).alias("prevented_fraud_value")
)

fraud_metrics_daily = fraud_metrics_daily.join(
    prevented_fraud_daily, "metric_date", "left"
).fillna(0)

# METRIC 13: Fraud detection lag
fraud_detection_lag = fraud_base.filter(
    (F.col("status") == "STATUS_82_CLOSED_CONFIRMED") &
    (F.col("event_ts").isNotNull())
).withColumn(
    "detection_lag_seconds",
    F.abs((F.unix_timestamp("case_created_ts") - F.unix_timestamp("event_ts")))
).groupBy("metric_date").agg(
    F.avg("detection_lag_seconds").alias("avg_detection_lag_seconds")
).withColumn(
    "avg_detection_lag_hours",
    F.round(F.col("avg_detection_lag_seconds") / 3600, 2)
).withColumn(
    "avg_detection_lag_days",
    F.round(F.col("avg_detection_lag_seconds") / (3600 * 24), 2)
)

fraud_metrics_daily = fraud_metrics_daily.join(
    fraud_detection_lag, "metric_date", "left"
)

# METRIC 14: Manual detection rate
manual_fraud = fraud_base.filter(
    (F.col("case_creation_type") != "AUTOMATIC_SYSTEM") &
    (F.col("status") == "STATUS_82_CLOSED_CONFIRMED")
).groupBy("metric_date").agg(
    F.count("case_id").alias("manual_detected_fraud")
)

fraud_metrics_daily = fraud_metrics_daily.join(
    manual_fraud, "metric_date", "left"
).fillna(0)

fraud_metrics_daily = fraud_metrics_daily.withColumn(
    "manual_detection_rate_pct",
    F.when(
        F.col("confirmed_cases") > 0,
        F.round((F.col("manual_detected_fraud") / F.col("confirmed_cases")) * 100, 2)
    ).otherwise(0)
)

# METRIC 15: Overall fraud rate
fraud_metrics_daily = fraud_metrics_daily.withColumn(
    "fraud_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.col("confirmed_cases") / F.col("transactions_evaluated")) * 100, 4)
    ).otherwise(0)
)

# METRIC 16: Fraud concentration by typology
fraud_by_typology = fraud_base.filter(
    F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
).groupBy("metric_date", "alert_type_norm").agg(
    F.count("case_id").alias("confirmed_by_typology"),
    F.sum(F.when(F.col("tx_amount").isNotNull(), F.col("tx_amount")).otherwise(0)).alias("fraud_value_by_typology")
)

# METRIC 17-18: Fraud values
fraud_metrics_daily = fraud_metrics_daily.withColumn(
    "avg_fraud_value",
    F.when(
        F.col("confirmed_cases") > 0,
        F.round(F.col("confirmed_fraud_value") / F.col("confirmed_cases"), 2)
    ).otherwise(0)
)

# METRIC 19: Fraud loss by typology (aggregated separately)
# METRIC 20: Refuted rate by typology (aggregated separately)
refuted_by_typology = fraud_base.filter(
    F.col("status") == "STATUS_81_CLOSED_REFUTED"
).groupBy("metric_date", "alert_type_norm").agg(
    F.count("case_id").alias("refuted_by_typology")
)

total_alerts_by_typology = alerts_with_time.groupBy("metric_date", "alert_type_norm").agg(
    F.countDistinct("alert_id").alias("total_alerts_by_typology")
)

refuted_rate_typology = refuted_by_typology.join(
    total_alerts_by_typology, ["metric_date", "alert_type_norm"], "left"
).withColumn(
    "refuted_rate_pct",
    F.when(
        F.col("total_alerts_by_typology") > 0,
        F.round((F.col("refuted_by_typology") / F.col("total_alerts_by_typology")) * 100, 2)
    ).otherwise(0)
)

# Add timestamp
fraud_metrics_daily = fraud_metrics_daily.withColumn(
    "metric_ts", F.lit(datetime.now())
)

print("\n✓ Fraud Metrics Calculated (20 metrics):")
for i, metric in enumerate([
    "Confirmed Cases",
    "Refuted Cases",
    "Inconclusive Cases",
    "True Positives",
    "False Positives",
    "False Negatives",
    "Precision",
    "Recall (TPR)",
    "F1 Score",
    "False Positive Rate",
    "False Negative Rate",
    "Prevented Fraud Value",
    "Fraud Detection Lag",
    "Manual Detection Rate",
    "Overall Fraud Rate",
    "Fraud Concentration by Typology",
    "Total Confirmed Fraud Value",
    "Average Fraud Value",
    "Fraud Loss by Typology",
    "Refuted Rate by Typology"
], 1):
    print(f"  Metric {i}: {metric}")

# Display sample data
print("\nSample Fraud Metrics (Daily Granularity):")
fraud_metrics_daily.select(
    "metric_date", "confirmed_cases", "refuted_cases", "inconclusive_cases",
    "precision", "recall_tpr", "f1_score", "fraud_rate_pct", "avg_fraud_value"
).limit(10).show(truncate=False)

print("\nFraud Metrics by Typology:")
fraud_by_typology.show(10, truncate=False)


2.4 FRAUD METRICS - GOLD LAYER

✓ Fraud Metrics Calculated (20 metrics):
  Metric 1: Confirmed Cases
  Metric 2: Refuted Cases
  Metric 3: Inconclusive Cases
  Metric 4: True Positives
  Metric 5: False Positives
  Metric 6: False Negatives
  Metric 7: Precision
  Metric 8: Recall (TPR)
  Metric 9: F1 Score
  Metric 10: False Positive Rate
  Metric 11: False Negative Rate
  Metric 12: Prevented Fraud Value
  Metric 13: Fraud Detection Lag
  Metric 14: Manual Detection Rate
  Metric 15: Overall Fraud Rate
  Metric 16: Fraud Concentration by Typology
  Metric 17: Total Confirmed Fraud Value
  Metric 18: Average Fraud Value
  Metric 19: Fraud Loss by Typology
  Metric 20: Refuted Rate by Typology

Sample Fraud Metrics (Daily Granularity):
+-----------+---------------+-------------+------------------+---------+----------+--------+--------------+---------------+
|metric_date|confirmed_cases|refuted_cases|inconclusive_cases|precision|recall_tpr|f1_score|fraud_rate_pct|avg_fraud_value|
+----

Transaction

In [21]:
transactions = spark.read.format('hudi').load(transactions_gold_path)

In [22]:
transactions.show(truncate=False)

+-------------------+----------------------+------------------+----------------------+---------------------------------------------------------------------------+--------------+-----------------------------------+---------+---------------+-----------------------------------+---------+---------+------+------------+------------+------------+-----------------------+--------------------------+------------------+--------------------------------------+----------------------------------------------------------------+----------+
|_hoodie_commit_time|_hoodie_commit_seqno  |_hoodie_record_key|_hoodie_partition_path|_hoodie_file_name                                                          |transaction_id|end_to_end_id                      |tenant_id|tx_type        |tx_msg_id                          |tx_status|tx_amount|tx_ccy|instg_mmb_id|instd_mmb_id|charge_count|event_ts               |ingested_at_ts            |event_to_ingest_ms|source_file_path                      |record_hash            

Transactions

In [24]:
# ============================================================================
# SECTION 2.1: TRANSACTION MONITORING METRICS
# Pre-Aggregated at Multiple Granularities (Hourly, Daily, Monthly, Quarterly, Annually)
# ============================================================================

print("\n" + "="*80)
print("2.1 TRANSACTION MONITORING METRICS - GOLD LAYER")
print("="*80)


def add_time_dimensions(df, timestamp_col="ingested_at_ts"):
    return df.withColumn("metric_date", F.to_date(F.col(timestamp_col))) \
             .withColumn("metric_hour", F.hour(F.col(timestamp_col))) \
             .withColumn("metric_month", F.month(F.col(timestamp_col))) \
             .withColumn("metric_year", F.year(F.col(timestamp_col))) \
             .withColumn("metric_quarter", F.quarter(F.col(timestamp_col))) 

# Prepare transactions with time dimensions
tx_with_time = add_time_dimensions(transactions, timestamp_col="ingested_at_ts")

# Join with alerts to identify evaluated transactions
tx_evaluated = tx_with_time.join(
    alerts.select("tx_msg_id").distinct(),
    tx_with_time.tx_msg_id == alerts.tx_msg_id,
    "inner"
).select(tx_with_time["*"])

# METRIC 1: Number of transactions received by TMS API
tms_received_daily = tx_with_time.groupBy(
    "metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year"
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_received"),
    F.count("transaction_id").alias("total_tx_records"),
    F.lit(datetime.now()).alias("metric_ts_tr")
).withColumn("metric_id", F.concat_ws("-", "metric_date", "transactions_received"))

# METRIC 2: Number of transactions evaluated
tms_evaluated_daily = tx_evaluated.groupBy(
    "metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year"
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_evaluated"),
    F.lit(datetime.now()).alias("metric_ts_te")
).withColumn("metric_id", F.concat_ws("-", "metric_date", "transactions_evaluated"))

# METRIC 3: Received vs Evaluated Rate (%)
tms_rate_daily = tms_received_daily.join(
    tms_evaluated_daily,
    ["metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year"],
    "left"
).withColumn(
    "received_vs_evaluated_rate_pct",
    F.when(
        F.col("transactions_received") > 0,
        F.round((F.col("transactions_evaluated") / F.col("transactions_received")) * 100, 2)
    ).otherwise(0)
).select(
    "metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year",
    "transactions_received", "transactions_evaluated", "received_vs_evaluated_rate_pct", "metric_ts_tr", "metric_ts_te",
    F.concat_ws("-", "metric_date", "received_vs_evaluated_rate_pct").alias("metric_id")
)

# METRIC 4: Transaction Evaluation Time (Average)
tms_evaluation_time = tx_with_time.join(
    alerts.select("tx_msg_id", F.col("created_at_ts").alias("alert_created_ts")),
    "tx_msg_id",
    "inner"
).withColumn(
    "evaluation_time_seconds",
    F.abs((F.unix_timestamp("alert_created_ts") - F.unix_timestamp("ingested_at_ts")))
).groupBy("metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year").agg(
    F.avg("evaluation_time_seconds").alias("avg_evaluation_time_seconds"),
    F.min("evaluation_time_seconds").alias("min_evaluation_time_seconds"),
    F.max("evaluation_time_seconds").alias("max_evaluation_time_seconds"),
    F.count("tx_msg_id").alias("evaluated_tx_count"),
    F.lit(datetime.now()).alias("metric_ts")
).withColumn(
    "metric_id", F.concat_ws("-", "metric_date", "avg_evaluation_time_seconds")
)

# Combine all TMS metrics for comprehensive view
tms_metrics_combined = tms_rate_daily.join(
    tms_evaluation_time,
    ["metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year"],
    "left"
)

print("\n✓ Transaction Monitoring Metrics Calculated:")
print(f"  - Metric 1: Transactions Received (daily)")
print(f"  - Metric 2: Transactions Evaluated (daily)")
print(f"  - Metric 3: Received vs Evaluated Rate (daily)")
print(f"  - Metric 4: Avg Evaluation Time (daily)")

# Display sample data
print("\nSample TMS Metrics (Daily Granularity):")
tms_metrics_combined.select(
    "metric_date", "transactions_received", "transactions_evaluated",
    "received_vs_evaluated_rate_pct", "avg_evaluation_time_seconds"
).limit(10).show(truncate=False)


2.1 TRANSACTION MONITORING METRICS - GOLD LAYER

✓ Transaction Monitoring Metrics Calculated:
  - Metric 1: Transactions Received (daily)
  - Metric 2: Transactions Evaluated (daily)
  - Metric 3: Received vs Evaluated Rate (daily)
  - Metric 4: Avg Evaluation Time (daily)

Sample TMS Metrics (Daily Granularity):
+-----------+---------------------+----------------------+------------------------------+---------------------------+
|metric_date|transactions_received|transactions_evaluated|received_vs_evaluated_rate_pct|avg_evaluation_time_seconds|
+-----------+---------------------+----------------------+------------------------------+---------------------------+
|2026-04-13 |307786               |111420                |36.2                          |82801.0                    |
+-----------+---------------------+----------------------+------------------------------+---------------------------+



# ALERT METRICS SECTION

In [25]:
# ============================================================================
# SECTION 2.2: ALERT METRICS - GOLD LAYER
# Pre-Aggregated at Multiple Granularities (Hourly, Daily, Monthly, Quarterly, Annually)
# ============================================================================

print("\n" + "="*80)
print("2.2 ALERT METRICS - GOLD LAYER")
print("="*80)

# Prepare alerts with time dimensions
alerts_with_time = add_time_dimensions(alerts, timestamp_col="created_at_ts")

# Join with transaction status info
alerts_with_tx = alerts_with_time.join(
    transactions.select("tx_msg_id", F.col("tx_status").alias("transaction_status")),
    "tx_msg_id",
    "left"
)

# METRIC 1: Number of transactions that generated an alert
alert_count_daily = alerts_with_time.groupBy(
    "metric_date", "metric_hour", "metric_month", "metric_quarter", "metric_year"
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_generated_alert"),
    F.countDistinct("alert_id").alias("total_alerts"),
    F.lit(datetime.now()).alias("metric_ts")
).withColumn("metric_id", F.concat_ws("-", "metric_date", "transactions_generated_alert"))

# METRIC 2: Received Alert Rate (%)
received_tx = transactions.groupBy(
    F.date_format(F.col("ingested_at_ts"), "yyyy-MM-dd").alias("metric_date")
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_received")
)

alert_rates = alert_count_daily.join(
    received_tx, "metric_date", "left"
).withColumn(
    "received_alert_rate_pct",
    F.when(
        F.col("transactions_received") > 0,
        F.round((F.col("transactions_generated_alert") / F.col("transactions_received")) * 100, 2)
    ).otherwise(0)
)

# METRIC 3: Evaluated Alert Rate (%)
evaluated_tx = tx_evaluated.groupBy(
    F.date_format(F.col("ingested_at_ts"), "yyyy-MM-dd").alias("metric_date")
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_evaluated")
)

alert_rates = alert_rates.join(
    evaluated_tx, "metric_date", "left"
).withColumn(
    "evaluated_alert_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.col("transactions_generated_alert") / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
)

# METRIC 4: AML Alert Rate
aml_alert_count = alerts_with_time.filter(
    (F.col("alert_type_norm").isin("AML", "FRAUD_AND_AML"))
).groupBy("metric_date").agg(
    F.countDistinct("tx_msg_id").alias("aml_alert_transactions")
)

alert_rates = alert_rates.join(
    aml_alert_count, "metric_date", "left"
).withColumn(
    "aml_alert_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.coalesce(F.col("aml_alert_transactions"), F.lit(0)) / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
)

# METRIC 5 & 7: Alert Velocity (Period-over-Period) - placeholder with lagged calculation
alert_rates_with_lag = alert_rates.withColumn(
    "prev_aml_alert_rate",
    F.lag("aml_alert_rate_pct").over(Window.orderBy("metric_date"))
).withColumn(
    "aml_alert_velocity_pct_change",
    F.when(
        F.col("prev_aml_alert_rate").isNotNull() & (F.col("prev_aml_alert_rate") != 0),
        F.round(((F.col("aml_alert_rate_pct") - F.col("prev_aml_alert_rate")) / F.col("prev_aml_alert_rate")) * 100, 2)
    ).otherwise(F.lit(None))
)

# METRIC 6: Fraud Alert Rate
fraud_alert_count = alerts_with_time.filter(
    (F.col("alert_type_norm").isin("FRAUD", "FRAUD_AND_AML"))
).groupBy("metric_date").agg(
    F.countDistinct("tx_msg_id").alias("fraud_alert_transactions")
)

alert_rates_with_fraud = alert_rates_with_lag.join(
    fraud_alert_count, "metric_date", "left"
).withColumn(
    "fraud_alert_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.coalesce(F.col("fraud_alert_transactions"), F.lit(0)) / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "prev_fraud_alert_rate",
    F.lag("fraud_alert_rate_pct").over(Window.orderBy("metric_date"))
).withColumn(
    "fraud_alert_velocity_pct_change",
    F.when(
        F.col("prev_fraud_alert_rate").isNotNull() & (F.col("prev_fraud_alert_rate") != 0),
        F.round(((F.col("fraud_alert_rate_pct") - F.col("prev_fraud_alert_rate")) / F.col("prev_fraud_alert_rate")) * 100, 2)
    ).otherwise(F.lit(None))
)

# METRIC 8: Interdiction Alerts (Blocked/Rejected transactions with alerts)
interdiction_alerts_count = alerts_with_tx.filter(
    F.col("transaction_status").isin("BLOCKED", "REJECTED")
).groupBy("metric_date").agg(
    F.countDistinct("alert_id").alias("interdiction_alerts")
)

# METRIC 9: Typology Alert Concentration
typology_concentration = alerts_with_time.groupBy(
    "metric_date", "alert_type_norm"
).agg(
    F.countDistinct("tx_msg_id").alias("transactions_by_typology"),
    F.count("alert_id").alias("alerts_by_typology")
)

# METRIC 10: Overall Interdiction Rate
alert_rates_with_fraud = alert_rates_with_fraud.join(
    interdiction_alerts_count, "metric_date", "left"
).withColumn(
    "overall_interdiction_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.coalesce(F.col("interdiction_alerts"), F.lit(0)) / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
)

# METRIC 11 & 12: Block-related metrics
block_alerts = alerts_with_tx.filter(
    F.col("transaction_status") == "BLOCKED"
).groupBy("metric_date").agg(
    F.countDistinct("alert_id").alias("block_condition_alerts"),
    F.countDistinct("tx_msg_id").alias("blocked_transactions")
)

alert_metrics_final = alert_rates_with_fraud.join(
    block_alerts, "metric_date", "left"
).withColumn(
    "block_condition_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.coalesce(F.col("blocked_transactions"), F.lit(0)) / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
)

# METRIC 13, 14, 15, 16: Override-related metrics
override_alerts_count = alerts_with_tx.filter(
    F.col("transaction_status") == "ACCC"
).groupBy("metric_date").agg(
    F.countDistinct("tx_msg_id").alias("override_transactions"),
    F.countDistinct("alert_id").alias("override_alerts")
)

alert_metrics_final = alert_metrics_final.join(
    override_alerts_count, "metric_date", "left"
).withColumn(
    "override_condition_rate_pct",
    F.when(
        F.col("transactions_evaluated") > 0,
        F.round((F.coalesce(F.col("override_transactions"), F.lit(0)) / F.col("transactions_evaluated")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "override_rate_pct",
    F.when(
        F.col("transactions_generated_alert") > 0,
        F.round((F.coalesce(F.col("override_transactions"), F.lit(0)) / F.col("transactions_generated_alert")) * 100, 2)
    ).otherwise(0)
).withColumn(
    "block_rate_pct",
    F.when(
        F.col("transactions_generated_alert") > 0,
        F.round((F.coalesce(F.col("blocked_transactions"), F.lit(0)) / F.col("transactions_generated_alert")) * 100, 2)
    ).otherwise(0)
)

print("\n✓ Alert Metrics Calculated (16 metrics):")
for i, metric in enumerate([
    "Transactions Generated Alert",
    "Received Alert Rate",
    "Evaluated Alert Rate",
    "AML Alert Rate",
    "AML Alert Velocity",
    "Fraud Alert Rate",
    "Fraud Alert Velocity",
    "Interdiction Alerts",
    "Typology Alert Concentration",
    "Overall Interdiction Rate",
    "Block Condition Alerts",
    "Block Condition Rate",
    "Override Alerts",
    "Override Condition Rate",
    "Override Rate",
    "Block Rate"
], 1):
    print(f"  Metric {i}: {metric}")

# Display sample data
print("\nSample Alert Metrics (Daily Granularity):")
alert_metrics_final.select(
    "metric_date", "transactions_generated_alert", "received_alert_rate_pct",
    "evaluated_alert_rate_pct", "aml_alert_rate_pct", "fraud_alert_rate_pct",
    "overall_interdiction_rate_pct", "block_rate_pct"
).limit(10).show(truncate=False)


2.2 ALERT METRICS - GOLD LAYER

✓ Alert Metrics Calculated (16 metrics):
  Metric 1: Transactions Generated Alert
  Metric 2: Received Alert Rate
  Metric 3: Evaluated Alert Rate
  Metric 4: AML Alert Rate
  Metric 5: AML Alert Velocity
  Metric 6: Fraud Alert Rate
  Metric 7: Fraud Alert Velocity
  Metric 8: Interdiction Alerts
  Metric 9: Typology Alert Concentration
  Metric 10: Overall Interdiction Rate
  Metric 11: Block Condition Alerts
  Metric 12: Block Condition Rate
  Metric 13: Override Alerts
  Metric 14: Override Condition Rate
  Metric 15: Override Rate
  Metric 16: Block Rate

Sample Alert Metrics (Daily Granularity):
+-----------+----------------------------+-----------------------+------------------------+------------------+--------------------+-----------------------------+--------------+
|metric_date|transactions_generated_alert|received_alert_rate_pct|evaluated_alert_rate_pct|aml_alert_rate_pct|fraud_alert_rate_pct|overall_interdiction_rate_pct|block_rate_pct|
+---

In [26]:
# ============================================================================
# SECTION 3: GOLD LAYER MATERIALIZATION & AGGREGATION
# Store all metrics at multiple granularities for dashboard consumption
# ============================================================================

print("\n" + "="*80)
print("3. METRICS GOLD LAYER MATERIALIZATION")
print("="*80)

# ============================================================================
# 3.1 TRANSACTION MONITORING METRICS STORAGE
# ============================================================================
print("\n[1/4] Storing Transaction Monitoring Metrics...")

tms_metrics_hourly = tms_metrics_combined.select(
    F.concat_ws("-", "metric_hour", F.monotonically_increasing_id()).alias("metric_id"),
    "metric_hour", "metric_month", "metric_quarter", "metric_year",
    "transactions_received", "transactions_evaluated", "received_vs_evaluated_rate_pct",
    "avg_evaluation_time_seconds", "min_evaluation_time_seconds", "max_evaluation_time_seconds",
    "metric_ts"
)

tms_metrics_daily = tms_metrics_combined.select(
    F.concat_ws("-", F.monotonically_increasing_id()).alias("metric_id"), "metric_month", "metric_quarter", "metric_year",
    F.first("transactions_received").over(Window.partitionBy("metric_date")).alias("transactions_received"),
    F.first("transactions_evaluated").over(Window.partitionBy("metric_date")).alias("transactions_evaluated"),
    F.first("received_vs_evaluated_rate_pct").over(Window.partitionBy("metric_date")).alias("received_vs_evaluated_rate_pct"),
    F.avg("avg_evaluation_time_seconds").over(Window.partitionBy("metric_date")).alias("avg_evaluation_time_seconds"),
    "metric_ts"
)

print("  ✓ TMS metrics prepared (hourly & daily)")

# ============================================================================
# 3.2 ALERT METRICS STORAGE
# ============================================================================
print("\n[2/4] Storing Alert Metrics...")

alert_metrics_daily_with_id = alert_metrics_final.withColumn(
    "metric_id", F.concat_ws("-", "metric_date", F.monotonically_increasing_id())
).withColumn(
    "metric_ts", F.lit(datetime.now())
)

print("  ✓ Alert metrics prepared (daily)")

# ============================================================================
# 3.3 CASE METRICS STORAGE
# ============================================================================
print("\n[3/4] Storing Case Metrics...")

case_metrics_daily_with_id = case_metrics_daily.withColumn(
    "metric_id", F.concat_ws("-", "metric_date", F.monotonically_increasing_id())
)

print("  ✓ Case metrics prepared (daily)")

# ============================================================================
# 3.4 FRAUD METRICS STORAGE
# ============================================================================
print("\n[4/4] Storing Fraud Metrics...")

fraud_metrics_daily_with_id = fraud_metrics_daily.withColumn(
    "metric_id", F.concat_ws("-", "metric_date", F.monotonically_increasing_id())
)

fraud_by_typology_with_id = fraud_by_typology.withColumn(
    "metric_id", F.concat_ws("-", "metric_date", "alert_type_norm", F.monotonically_increasing_id())
).withColumn(
    "metric_ts", F.lit(datetime.now())
)

refuted_rate_typology_with_id = refuted_rate_typology.withColumn(
    "metric_id", F.concat_ws("-", "metric_date", "alert_type_norm", F.monotonically_increasing_id())
).withColumn(
    "metric_ts", F.lit(datetime.now())
)

print("  ✓ Fraud metrics prepared (daily & by typology)")

# ============================================================================
# 3.5 SUMMARY STATISTICS
# ============================================================================
print("\n" + "="*80)
print("METRICS SUMMARY")
print("="*80)

print(f"\n📊 TRANSACTION MONITORING METRICS:")
print(f"   Total Daily Records: {tms_metrics_daily.count()}")
print(f"   Metrics Calculated: 4 (Received, Evaluated, Rate, Evaluation Time)")
print(f"   Time Granularities: Hourly, Daily, Monthly, Quarterly, Annually")

print(f"\n📊 ALERT METRICS:")
print(f"   Total Daily Records: {alert_metrics_daily_with_id.count()}")
print(f"   Metrics Calculated: 16 (Alert counts, rates, velocities, interdictions)")
print(f"   Time Granularities: Daily, Monthly, Quarterly, Annually")

print(f"\n📊 CASE METRICS:")
print(f"   Total Daily Records: {case_metrics_daily_with_id.count()}")
print(f"   Metrics Calculated: 23 (Cases, rates, durations, tasks, dispositions)")
print(f"   Time Granularities: Daily, Monthly, Quarterly, Annually")

print(f"\n📊 FRAUD METRICS:")
print(f"   Total Daily Records: {fraud_metrics_daily_with_id.count()}")
print(f"   Typology Records: {fraud_by_typology_with_id.count()}")
print(f"   Refuted Rate Records: {refuted_rate_typology_with_id.count()}")
print(f"   Metrics Calculated: 20 (Outcomes, accuracy, performance, rates, values)")
print(f"   Time Granularities: Daily, Monthly, Quarterly, Annually")

print("\n" + "="*80)
print("METRICS READINESS FOR DASHBOARD CONSUMPTION")
print("="*80)
print("\n✓ All metrics are pre-aggregated in gold layer")
print("✓ Multiple time granularities supported")
print("✓ Ready for BIAR dashboard consumption without transformation")
print("✓ Standardized metric IDs and timestamps for lineage tracking")
print("✓ Logical grouping: TMS, Alert, Case, and Fraud metrics")


3. METRICS GOLD LAYER MATERIALIZATION

[1/4] Storing Transaction Monitoring Metrics...
  ✓ TMS metrics prepared (hourly & daily)

[2/4] Storing Alert Metrics...
  ✓ Alert metrics prepared (daily)

[3/4] Storing Case Metrics...
  ✓ Case metrics prepared (daily)

[4/4] Storing Fraud Metrics...
  ✓ Fraud metrics prepared (daily & by typology)

METRICS SUMMARY

📊 TRANSACTION MONITORING METRICS:
   Total Daily Records: 1
   Metrics Calculated: 4 (Received, Evaluated, Rate, Evaluation Time)
   Time Granularities: Hourly, Daily, Monthly, Quarterly, Annually

📊 ALERT METRICS:
   Total Daily Records: 1
   Metrics Calculated: 16 (Alert counts, rates, velocities, interdictions)
   Time Granularities: Daily, Monthly, Quarterly, Annually

📊 CASE METRICS:
   Total Daily Records: 2
   Metrics Calculated: 23 (Cases, rates, durations, tasks, dispositions)
   Time Granularities: Daily, Monthly, Quarterly, Annually

📊 FRAUD METRICS:
   Total Daily Records: 0
   Typology Records: 0
   Refuted Rate Record

In [27]:
# ============================================================================
# SECTION 4: DASHBOARD CONSUMPTION EXAMPLES
# Demonstrates how BIAR dashboards consume pre-aggregated metrics
# ============================================================================

print("\n" + "="*80)
print("4. BIAR DASHBOARD CONSUMPTION EXAMPLES")
print("="*80)

# ============================================================================
# EXAMPLE 1: TRANSACTION MONITORING DASHBOARD
# ============================================================================
print("\n\n### EXAMPLE 1: Transaction Monitoring Dashboard ###\n")

# Dashboard filters by date range and time granularity
dashboard_date = "2024-01-15"
tms_dashboard_data = tms_metrics_daily.filter(
    F.col("metric_date") == dashboard_date
).select(
    "transactions_received",
    "transactions_evaluated",
    "received_vs_evaluated_rate_pct",
    "avg_evaluation_time_seconds"
)

print("TMS Dashboard Input (No Transformation Needed):")
tms_dashboard_data.show(truncate=False)

# ============================================================================
# EXAMPLE 2: ALERT MONITORING DASHBOARD
# ============================================================================
print("\n\n### EXAMPLE 2: Alert Monitoring Dashboard ###\n")

alert_dashboard_data = alert_metrics_daily_with_id.filter(
    F.col("metric_date") == dashboard_date
).select(
    "metric_date",
    "transactions_generated_alert",
    "received_alert_rate_pct",
    "evaluated_alert_rate_pct",
    "aml_alert_rate_pct",
    "fraud_alert_rate_pct",
    "overall_interdiction_rate_pct",
    "block_rate_pct",
    "override_rate_pct"
)

print("Alert Dashboard Input (No Transformation Needed):")
alert_dashboard_data.show(truncate=False)

# ============================================================================
# EXAMPLE 3: CASE MANAGEMENT DASHBOARD
# ============================================================================
print("\n\n### EXAMPLE 3: Case Management Dashboard ###\n")

case_dashboard_data = case_metrics_daily.filter(
    F.col("metric_date") == dashboard_date
).select(
    "metric_date",
    "total_cases_created",
    "cases_from_alerts",
    "manual_cases_created",
    "auto_closed_cases",
    "total_cases_closed",
    "confirmed_cases",
    "refuted_cases",
    "inconclusive_cases",
    "auto_closure_rate_pct",
    "confirmation_rate_pct",
    "avg_mttd_days",
    "open_tasks"
)

print("Case Management Dashboard Input (No Transformation Needed):")
case_dashboard_data.show(truncate=False)

# ============================================================================
# EXAMPLE 4: FRAUD ANALYTICS DASHBOARD
# ============================================================================
print("\n\n### EXAMPLE 4: Fraud Analytics Dashboard ###\n")

fraud_dashboard_data = fraud_metrics_daily.filter(
    F.col("metric_date") == dashboard_date
).select(
    "metric_date",
    "confirmed_cases",
    "refuted_cases",
    "inconclusive_cases",
    "true_positives",
    "false_positives",
    "false_negatives",
    "precision",
    "recall_tpr",
    "f1_score",
    "fraud_rate_pct",
    "avg_fraud_value",
    "confirmed_fraud_value",
    "prevented_fraud_value",
    "avg_detection_lag_hours"
)

print("Fraud Analytics Dashboard Input (No Transformation Needed):")
fraud_dashboard_data.show(truncate=False)

# ============================================================================
# EXAMPLE 5: EXECUTIVE OVERVIEW DASHBOARD (Multi-Metric KPIs)
# ============================================================================
print("\n\n### EXAMPLE 5: Executive Overview Dashboard (Multi-Metric KPIs) ###\n")

executive_kpis = tms_metrics_daily.filter(
    F.col("metric_date") == dashboard_date
).select("metric_month").join(
    alert_metrics_daily_with_id.filter(F.col("metric_date") == dashboard_date).select("metric_date"),
    F.col("metric_date") == F.col("metric_date"),
    "inner"
).select(
    F.coalesce(F.col("metric_date"), F.col("metric_date")).alias("dashboard_date")
).crossJoin(
    tms_metrics_daily.filter(F.col("metric_date") == dashboard_date)
        .agg(F.first("transactions_received").alias("total_transactions_received"))
).crossJoin(
    alert_metrics_daily_with_id.filter(F.col("metric_date") == dashboard_date)
        .agg(F.first("transactions_generated_alert").alias("total_alerts"))
).crossJoin(
    case_metrics_daily.filter(F.col("metric_date") == dashboard_date)
        .agg(F.first("total_cases_created").alias("total_cases"))
).crossJoin(
    fraud_metrics_daily.filter(F.col("metric_date") == dashboard_date)
        .agg(F.first("confirmed_cases").alias("confirmed_fraud"))
).select(
    "dashboard_date",
    "total_transactions_received",
    "total_alerts",
    "total_cases",
    "confirmed_fraud"
)

print("Executive Overview Dashboard:")
executive_kpis.show(truncate=False)

# ============================================================================
# EXAMPLE 6: TIME-SERIES TRENDING (Monthly Aggregation)
# ============================================================================
print("\n\n### EXAMPLE 6: Time-Series Trending (Monthly Aggregation) ###\n")

monthly_fraud_trend = fraud_metrics_daily.filter(
    F.col("metric_date").isNotNull()
).groupBy(
    F.date_format(F.col("metric_date"), "yyyy-MM").alias("month")
).agg(
    F.sum("confirmed_cases").alias("monthly_confirmed_fraud"),
    F.avg("fraud_rate_pct").alias("monthly_avg_fraud_rate"),
    F.sum("confirmed_fraud_value").alias("monthly_fraud_value")
).orderBy("month")

print("Monthly Fraud Trend (No Transformation Needed - Pre-Aggregated):")
monthly_fraud_trend.show(truncate=False)

# ============================================================================
# EXAMPLE 7: TYPOLOGY BREAKDOWN (For Drill-Down Analysis)
# ============================================================================
print("\n\n### EXAMPLE 7: Typology Breakdown Dashboard ###\n")

typology_data = fraud_by_typology.filter(
    F.col("metric_date") == dashboard_date
).select(
    "metric_date",
    "alert_type_norm",
    "confirmed_by_typology",
    "fraud_value_by_typology"
)

print("Fraud Metrics by Typology (No Transformation Needed):")
typology_data.show(truncate=False)

print("\n\n" + "="*80)
print("KEY BENEFITS OF PRE-AGGREGATED GOLD LAYER METRICS")
print("="*80)
print("""
✓ NO REAL-TIME AGGREGATION: All metrics pre-calculated and stored
✓ INSTANT DASHBOARD RENDERING: Direct consumption without queries
✓ MULTIPLE GRANULARITIES: Same metrics at hourly/daily/monthly/quarterly/annual
✓ CONSISTENT DEFINITIONS: Single source of truth for all metrics
✓ EFFICIENT STORAGE: Optimized Hudi tables with partitioning
✓ SCALABLE QUERIES: Dashboard queries execute in milliseconds
✓ AUDIT TRAIL: Metric IDs and timestamps for lineage tracking
✓ LOGICAL GROUPING: TMS, Alert, Case, and Fraud metrics clearly separated
✓ METRIC FLEXIBILITY: Add new metrics without impacting dashboard layers
✓ STANDARDIZATION: All metrics follow same aggregation patterns
""")


4. BIAR DASHBOARD CONSUMPTION EXAMPLES


### EXAMPLE 1: Transaction Monitoring Dashboard ###

TMS Dashboard Input (No Transformation Needed):
+---------------------+----------------------+------------------------------+---------------------------+
|transactions_received|transactions_evaluated|received_vs_evaluated_rate_pct|avg_evaluation_time_seconds|
+---------------------+----------------------+------------------------------+---------------------------+
+---------------------+----------------------+------------------------------+---------------------------+



### EXAMPLE 2: Alert Monitoring Dashboard ###

Alert Dashboard Input (No Transformation Needed):
+-----------+----------------------------+-----------------------+------------------------+------------------+--------------------+-----------------------------+--------------+-----------------+
|metric_date|transactions_generated_alert|received_alert_rate_pct|evaluated_alert_rate_pct|aml_alert_rate_pct|fraud_alert_rate_pct|overall

26/04/17 05:15:50 WARN Column: Constructing trivially true equals predicate, ''metric_date = 'metric_date'. Perhaps you need to use aliases.


Executive Overview Dashboard:
+--------------+---------------------------+------------+-----------+---------------+
|dashboard_date|total_transactions_received|total_alerts|total_cases|confirmed_fraud|
+--------------+---------------------------+------------+-----------+---------------+
+--------------+---------------------------+------------+-----------+---------------+



### EXAMPLE 6: Time-Series Trending (Monthly Aggregation) ###

Monthly Fraud Trend (No Transformation Needed - Pre-Aggregated):
+-----+-----------------------+----------------------+-------------------+
|month|monthly_confirmed_fraud|monthly_avg_fraud_rate|monthly_fraud_value|
+-----+-----------------------+----------------------+-------------------+
+-----+-----------------------+----------------------+-------------------+



### EXAMPLE 7: Typology Breakdown Dashboard ###

Fraud Metrics by Typology (No Transformation Needed):
+-----------+---------------+---------------------+-----------------------+
|metric_date|

In [28]:
# ============================================================================
# SECTION 5: WRITE ALL METRICS TO GOLD LAYER (PARQUET FORMAT)
# Stores all pre-aggregated metrics for dashboard consumption
# ============================================================================

print("\n" + "="*80)
print("5. WRITING METRICS TO GOLD LAYER")
print("="*80)

# ============================================================================
# 1. WRITE TMS METRICS
# ============================================================================
print("\n### Writing Transaction Monitoring (TMS) Metrics ###")
try:
    tms_metrics_daily.write.mode("overwrite").parquet(tms_metrics_path)
    print(f"  ✓ {tms_metrics_daily.count()} TMS metric rows written")
    print(f"    Format: Parquet")
    print(f"    Path: {tms_metrics_path}")
except Exception as e:
    print(f"  ✗ Error writing TMS metrics: {e}")

# ============================================================================
# 2. WRITE ALERT METRICS
# ============================================================================
print("\n### Writing Alert Metrics ###")
try:
    alert_metrics_daily_with_id.write.mode("overwrite").parquet(alert_metrics_path)
    print(f"  ✓ {alert_metrics_daily_with_id.count()} Alert metric rows written")
    print(f"    Format: Parquet")
    print(f"    Path: {alert_metrics_path}")
except Exception as e:
    print(f"  ✗ Error writing Alert metrics: {e}")

# ============================================================================
# 3. WRITE CASE METRICS
# ============================================================================
print("\n### Writing Case Management Metrics ###")
try:
    case_metrics_daily_with_id.write.mode("overwrite").parquet(case_metrics_path)
    print(f"  ✓ {case_metrics_daily_with_id.count()} Case metric rows written")
    print(f"    Format: Parquet")
    print(f"    Path: {case_metrics_path}")
except Exception as e:
    print(f"  ✗ Error writing Case metrics: {e}")

# ============================================================================
# 4. WRITE FRAUD METRICS
# ============================================================================
print("\n### Writing Fraud Detection Metrics ###")
try:
    fraud_metrics_daily_with_id.write.mode("overwrite").parquet(fraud_metrics_path)
    print(f"  ✓ {fraud_metrics_daily_with_id.count()} Fraud metric rows written")
    print(f"    Format: Parquet")
    print(f"    Path: {fraud_metrics_path}")
except Exception as e:
    print(f"  ✗ Error writing Fraud metrics: {e}")

# ============================================================================
# VERIFICATION: Read back stored metrics
# ============================================================================
print("\n" + "="*80)
print("VERIFICATION: Stored Metrics Available for Dashboard")
print("="*80)

metrics_available = []

# Verify TMS metrics
try:
    tms_verify = spark.read.parquet(tms_metrics_path)
    tms_count = tms_verify.count()
    print(f"\n✓ TMS Metrics: {tms_count} rows available")
    print("  Sample:")
    tms_verify.select("metric_month", "transactions_received", "transactions_evaluated").limit(2).show()
    metrics_available.append(("TMS", tms_metrics_path, tms_count))
except Exception as e:
    print(f"✗ TMS Metrics: {e}")

# Verify Alert metrics
try:
    alert_verify = spark.read.parquet(alert_metrics_path)
    alert_count = alert_verify.count()
    print(f"\n✓ Alert Metrics: {alert_count} rows available")
    print("  Sample:")
    alert_verify.select("metric_month", "transactions_generated_alert", "evaluated_alert_rate_pct").limit(2).show()
    metrics_available.append(("Alert", alert_metrics_path, alert_count))
except Exception as e:
    print(f"✗ Alert Metrics: {e}")

# Verify Case metrics
try:
    case_verify = spark.read.parquet(case_metrics_path)
    case_count = case_verify.count()
    print(f"\n✓ Case Metrics: {case_count} rows available")
    print("  Sample:")
    case_verify.select("metric_month", "total_cases_created", "confirmed_cases").limit(2).show()
    metrics_available.append(("Case", case_metrics_path, case_count))
except Exception as e:
    print(f"✗ Case Metrics: {e}")

# Verify Fraud metrics
try:
    fraud_verify = spark.read.parquet(fraud_metrics_path)
    fraud_count = fraud_verify.count()
    print(f"\n✓ Fraud Metrics: {fraud_count} rows available")
    print("  Sample:")
    fraud_verify.select("metric_month", "confirmed_cases", "fraud_rate_pct").limit(2).show()
    metrics_available.append(("Fraud", fraud_metrics_path, fraud_count))
except Exception as e:
    print(f"✗ Fraud Metrics: {e}")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✓ ALL METRICS SUCCESSFULLY WRITTEN TO GOLD LAYER")
print("="*80)

if metrics_available:
    total_rows = sum(count for _, _, count in metrics_available)
    print(f"\n✓ {len(metrics_available)} metrics written ({total_rows:,} total rows):\n")
    for metric_type, path, count in metrics_available:
        print(f"   {metric_type:8} → {count:6,} rows at {path.split('/')[-1]}/")
    
    print(f"""
DASHBOARD INTEGRATION:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

All metrics are now stored and ready for Executive Dashboard (db_us2.ipynb):

  TMS Metrics        → {tms_metrics_path}
  Alert Metrics      → {alert_metrics_path}
  Case Metrics       → {case_metrics_path}
  Fraud Metrics      → {fraud_metrics_path}

Dashboard Section 2 will read these Parquet tables to:
  ✓ Calculate fraud detection KPIs
  ✓ Compute operational efficiency metrics
  ✓ Analyze financial impact
  ✓ Generate trend indicators
  ✓ Render interactive visualizations
  ✓ Export results in JSON/CSV/HTML formats

You can now run db_us2.ipynb to generate executive KPI reports.
""")
else:
    print("\n✗ No metrics were successfully written. Check errors above.")



5. WRITING METRICS TO GOLD LAYER

### Writing Transaction Monitoring (TMS) Metrics ###
  ✓ 1 TMS metric rows written
    Format: Parquet
    Path: /opt/Tazama_Warehouse/metrics/transaction_monitoring

### Writing Alert Metrics ###


26/04/17 05:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 05:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 05:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 05:15:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 05:15:59 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/17 05:16:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradat

  ✓ 1 Alert metric rows written
    Format: Parquet
    Path: /opt/Tazama_Warehouse/metrics/alerts

### Writing Case Management Metrics ###
  ✓ 2 Case metric rows written
    Format: Parquet
    Path: /opt/Tazama_Warehouse/metrics/cases

### Writing Fraud Detection Metrics ###
  ✓ 0 Fraud metric rows written
    Format: Parquet
    Path: /opt/Tazama_Warehouse/metrics/fraud

VERIFICATION: Stored Metrics Available for Dashboard

✓ TMS Metrics: 1 rows available
  Sample:
+------------+---------------------+----------------------+
|metric_month|transactions_received|transactions_evaluated|
+------------+---------------------+----------------------+
|           4|               307786|                111420|
+------------+---------------------+----------------------+


✓ Alert Metrics: 1 rows available
  Sample:
+------------+----------------------------+------------------------+
|metric_month|transactions_generated_alert|evaluated_alert_rate_pct|
+------------+----------------------------+